# Online Retail II — cleaning & EDA

Prep notes for the Retail Executive Dashboard.
Combine both Excel years, drop junk lines, then check revenue / orders / customers against the Power BI cards (~£20M, ~40K orders, ~5.9K customers).


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data")
OUT = Path("../python/outputs")
OUT.mkdir(parents=True, exist_ok=True)

r1 = pd.read_excel(DATA / "online_retail_II.xlsx", sheet_name="Year 2009-2010")
r2 = pd.read_excel(DATA / "online_retail_II.xlsx", sheet_name="Year 2010-2011")
raw = pd.concat([r1, r2], ignore_index=True)
print(raw.shape)
print(raw.isna().sum())
raw.head(3)


In [ ]:
# duplicates + cancel / fee codes
print("exact dups:", raw.duplicated().sum())
df = raw.drop_duplicates().copy()
df["Invoice"] = df["Invoice"].astype(str)
print("cancel rows:", df["Invoice"].str.startswith("C").sum())
fee_codes = ["POST", "DOT", "M", "D", "AMAZONFEE", "BANK CHARGES", "CRUK"]
print(df["StockCode"].astype(str).isin(fee_codes).value_counts())


In [ ]:
# revenue line set (same spirit as Power Query rules)
rev = df[
    (~df["Invoice"].str.startswith("C"))
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
    & (df["Description"].notna())
    & (~df["StockCode"].astype(str).isin(fee_codes))
].copy()
rev["LineAmount"] = rev["Quantity"] * rev["Price"]

revenue = rev["LineAmount"].sum()
orders = rev["Invoice"].nunique()
customers = rev["Customer ID"].nunique(dropna=True)
aov = revenue / orders
print(f"revenue £{revenue/1e6:.2f}M")
print(f"orders  {orders:,}")
print(f"customers {customers:,}")
print(f"AOV     £{aov:.2f}")


In [ ]:
# monthly orders trend
rev["Month"] = pd.to_datetime(rev["InvoiceDate"]).dt.to_period("M").dt.to_timestamp()
monthly = rev.groupby("Month").agg(orders=("Invoice", "nunique"), revenue=("LineAmount", "sum"))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(monthly.index, monthly["orders"], color="#2E7D32")
ax.set_title("Orders by month")
ax.set_ylabel("Orders")
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(OUT / "orders_by_month.png", dpi=120)
plt.show()
monthly.tail()


In [ ]:
# top products / countries
print(rev.groupby("Description")["Invoice"].nunique().sort_values(ascending=False).head(10))
print(rev.groupby("Country")["Invoice"].nunique().sort_values(ascending=False).head(10))


In [ ]:
# guest share of revenue
rev["IsGuest"] = rev["Customer ID"].isna()
print(rev.groupby("IsGuest").agg(lines=("Invoice", "size"), revenue=("LineAmount", "sum"), orders=("Invoice", "nunique")))
